In [ ]:
"""
Almgren-Chriss Optimal Execution Model
=======================================
Paper : Almgren & Chriss (2000) "Optimal Execution of Portfolio Transactions"
        Journal of Risk, Vol. 3 No. 2

PART 1  Faithful replication of the closed-form AC model
PART 2  Extension: Stochastic Volatility
        Relaxes the constant-sigma assumption by replacing the fixed
        volatility with a mean-reverting CIR variance process (Heston-style).
        At each slice the trader re-solves the AC sub-problem using the
        current variance estimate, giving a vol-adaptive execution schedule.

Parameters calibrated so that a 100 k-share block over one trading day
costs roughly 15 bps in expected implementation shortfall.
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

#  GLOBAL PARAMETERS

X0      = 1_000_000          # total shares to liquidate
S0      = 50.0               # arrival mid-price ($)
ETA     = 2.976e-10          # temporary impact (calibrated for ~15 bps)
GAMMA   = 1.5e-11            # permanent impact
SIGMA   = 0.25               # annual volatility (constant, AC base case)
T_YR    = 1 / 252            # horizon: 1 trading day expressed in years
N       = 20                 # number of equal time slices
DT      = T_YR / N
LAMBDA  = 1e-6               # risk aversion parameter


#  PART 1 -- ORIGINAL ALMGREN-CHRISS MODEL


class AlmgrenChriss:
    """
    Closed-form optimal liquidation under linear, constant market impact.

    Objective:  min  E[IS] + lambda * Var[IS]

    Optimal trajectory:
        x(t) = X0 * sinh(kappa*(T-t)) / sinh(kappa*T)

    where kappa = sqrt( lambda * sigma_step^2 / eta )
    """

    def __init__(self, X0=X0, S0=S0, eta=ETA, gamma=GAMMA,
                 sigma=SIGMA, T=T_YR, N=N, lam=LAMBDA):
        self.X0    = X0
        self.S0    = S0
        self.eta   = eta
        self.gamma = gamma
        self.T     = T
        self.N     = N
        self.dt    = T / N
        self.lam   = lam
        # per-step price volatility in dollars
        self.sigma_step = sigma * S0 * np.sqrt(self.dt)
        self.kappa      = np.sqrt(lam * self.sigma_step**2 / eta)

    def trajectory(self):
        """Returns (time_grid_years, shares_remaining)."""
        t     = np.linspace(0, self.T, self.N + 1)
        denom = np.sinh(self.kappa * self.T)
        if denom < 1e-12:
            x = self.X0 * (1 - t / self.T)
        else:
            x = self.X0 * np.sinh(self.kappa * (self.T - t)) / denom
        return t, np.maximum(x, 0.0)

    def trade_list(self):
        """Shares sold at each of the N slices."""
        _, x = self.trajectory()
        return np.diff(-x)          # positive = shares sold

    def expected_cost(self):
        """Analytical E[IS] in dollars."""
        k, T, X0, eta, gamma = self.kappa, self.T, self.X0, self.eta, self.gamma
        if k * T < 1e-8:
            return 0.5 * gamma * X0**2 + eta * X0**2 / T
        return (0.5 * gamma * X0**2
                + eta * X0**2 * k / (2 * np.tanh(k * T / 2)))

    def cost_variance(self):
        """Analytical Var[IS] in dollars^2."""
        k, T, X0 = self.kappa, self.T, self.X0
        s2       = self.sigma_step**2
        if k * T < 1e-8:
            return s2 * X0**2 * T / 3
        return (s2 * X0**2
                * (k * T / 2 - 0.5 * np.tanh(k * T / 2))
                / (k * np.sinh(k * T)))

    def simulate_paths(self, n_paths=500, seed=42):
        """
        Monte Carlo: forward-simulate the AC schedule.
        IS = total execution cost minus fair-value (X0 * arrival price).
        """
        rng    = np.random.default_rng(seed)
        trades = self.trade_list()
        pnl    = np.zeros(n_paths)
        for p in range(n_paths):
            price    = self.S0
            total    = 0.0
            for j, n in enumerate(trades):
                dW         = rng.standard_normal() * self.sigma_step
                price     += self.gamma * n + dW   # permanent shift + noise
                exec_px    = price + self.eta * n / self.dt   # temp impact
                total     += n * exec_px
            pnl[p] = total - self.X0 * self.S0    # IS = overpayment vs arrival
        return pnl

    def efficient_frontier(self, n_points=80):
        """Trace cost-variance frontier by sweeping lambda."""
        lam_grid = np.logspace(-9, -4, n_points)
        costs, variances = [], []
        for l in lam_grid:
            m = AlmgrenChriss(self.X0, self.S0, self.eta, self.gamma,
                              SIGMA, self.T, self.N, l)
            costs.append(m.expected_cost())
            variances.append(m.cost_variance())
        return np.array(costs), np.array(variances)

    def impact_decomposition(self):
        """Per-slice permanent and temporary impact in bps."""
        trades = self.trade_list()
        perm   = self.gamma * trades / self.S0 * 1e4
        temp   = self.eta * (trades / self.dt) / self.S0 * 1e4
        return perm, temp

#  PART 2 -- EXTENSION: STOCHASTIC VOLATILITY
#  KEY ASSUMPTION RELAXED:
#  AC (2000) requires sigma to be constant throughout the horizon.
#  In practice, intraday volatility varies substantially -- it is
#  well-documented to be high at open, low at midday, high at close
#  (the "U-shape"), and subject to sudden spikes on news arrival.
#
#  This extension replaces the fixed sigma with a CIR mean-reverting
#  variance process:
#
#    V_{j+1} = V_j + kappa_v*(theta_v - V_j)*dt + xi*sqrt(V_j*dt)*Z_j
#
#  At every slice j, the trader greedily re-solves the AC problem using
#  the current variance V_j and remaining inventory x_j, yielding a
#  locally optimal trade n_j. This produces a vol-adaptive schedule:
#  the trader speeds up when volatility spikes and slows down when
#  markets quieten.
# ============================================================

class StochasticVolAC:
    """
    Almgren-Chriss with greedy stochastic-vol re-optimisation.
    """

    def __init__(self, X0=X0, S0=S0, eta=ETA, gamma=GAMMA,
                 T=T_YR, N=N, lam=LAMBDA,
                 kappa_v=8.0, theta_v=None, xi=0.4, seed=42):
        self.X0      = X0
        self.S0      = S0
        self.eta     = eta
        self.gamma   = gamma
        self.T       = T
        self.N       = N
        self.dt      = T / N
        self.lam     = lam
        self.kappa_v = kappa_v
        self.theta_v = theta_v if theta_v is not None else SIGMA**2
        self.xi      = xi
        self.V0      = SIGMA**2
        self.rng     = np.random.default_rng(seed)

    def _local_kappa(self, V):
        """AC urgency parameter at current instantaneous variance V."""
        sigma_step = np.sqrt(V) * self.S0 * np.sqrt(self.dt)
        return np.sqrt(max(self.lam * sigma_step**2 / self.eta, 0.0))

    def _local_trade(self, x_rem, V, steps_left):
        """Greedy optimal trade: first step of AC sub-problem."""
        if steps_left <= 1:
            return x_rem
        T_rem = steps_left * self.dt
        k     = self._local_kappa(V)
        if k * T_rem < 1e-8:
            return x_rem / steps_left
        n = x_rem * (1 - np.sinh(k * (T_rem - self.dt)) / np.sinh(k * T_rem))
        return max(float(n), 0.0)

    def _step_variance(self, V):
        """One CIR step."""
        dW = self.rng.standard_normal()
        dV = (self.kappa_v * (self.theta_v - V) * self.dt
              + self.xi * np.sqrt(max(V, 0) * self.dt) * dW)
        return max(V + dV, 1e-12)

    def simulate(self, n_paths=500):
        """
        Returns
        -------
        trajectories : (n_paths, N+1)
        vol_paths    : (n_paths, N+1)   annualised sigma at each step
        costs        : (n_paths,)       IS in dollars
        """
        trajectories = np.zeros((n_paths, self.N + 1))
        vol_paths    = np.zeros((n_paths, self.N + 1))
        costs        = np.zeros(n_paths)

        for p in range(n_paths):
            x = float(self.X0)
            V = self.V0
            price = self.S0
            trajectories[p, 0] = x
            vol_paths[p, 0]    = np.sqrt(V)

            for j in range(self.N):
                n = min(self._local_trade(x, V, self.N - j), x)
                sigma_step = np.sqrt(V) * self.S0 * np.sqrt(self.dt)
                dW         = self.rng.standard_normal() * sigma_step
                price     += self.gamma * n + dW
                exec_px    = price + self.eta * n / self.dt
                costs[p]  += n * exec_px
                x         -= n
                V          = self._step_variance(V)
                trajectories[p, j+1] = x
                vol_paths[p, j+1]    = np.sqrt(V)

            if x > 0:
                costs[p] += x * price   # liquidate residual at market

        costs -= self.X0 * self.S0      # convert to IS
        return trajectories, vol_paths, costs

    def mean_trajectory(self, n_paths=500):
        trajs, vols, costs = self.simulate(n_paths)
        return trajs.mean(0), vols.mean(0), costs



#  VISUALISATION

C = {
    "blue"  : "#185FA5",
    "teal"  : "#0F6E56",
    "coral" : "#D85A30",
    "amber" : "#BA7517",
    "gray"  : "#5F5E5A",
    "lgray" : "#D3D1C7",
    "purple": "#534AB7",
    "red"   : "#A32D2D",
}

def plot_all():
    ac  = AlmgrenChriss()
    svc = StochasticVolAC(xi=0.4, kappa_v=8.0, seed=7)

    t_grid = np.linspace(0, T_YR, N + 1)
    t_hrs  = t_grid * 252 * 6.5

    _, x_ac   = ac.trajectory()
    x_twap    = X0 * (1 - np.linspace(0, 1, N + 1))
    x_sv, sigma_sv, sv_costs = svc.mean_trajectory(n_paths=800)

    perm_bps, temp_bps = ac.impact_decomposition()
    e_costs, e_vars    = ac.efficient_frontier()

    mc_costs_all = ac.simulate_paths(n_paths=600)
    _, _, sv_costs_all = svc.simulate(n_paths=600)

    fig = plt.figure(figsize=(18, 14), facecolor="white")
    fig.suptitle(
        "Almgren-Chriss Optimal Execution  |  Replication + Stochastic Vol Extension",
        fontsize=15, fontweight="bold", y=0.985, color="#2C2C2A"
    )

    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.48, wspace=0.38)

    def style(ax, title, xlabel, ylabel):
        ax.set_title(title, fontsize=11, fontweight="bold", pad=6)
        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.grid(True, alpha=0.22, lw=0.7)
        ax.tick_params(labelsize=8)

    # 1. Execution trajectories
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(t_hrs, x_ac / 1e6,   color=C["blue"],   lw=2.2, label="AC optimal")
    ax1.plot(t_hrs, x_twap / 1e6, color=C["red"],    lw=1.4, ls="--", alpha=0.8, label="TWAP")
    ax1.plot(t_hrs, x_sv / 1e6,   color=C["teal"],   lw=2.0, ls="-.", label="Stoch-vol AC")
    style(ax1, "Execution trajectory", "trading hour", "shares remaining (M)")
    ax1.legend(fontsize=7.5, framealpha=0)
    ax1.set_xlim(0, t_hrs[-1])
    ax1.set_ylim(0, 1.05)

    # 2. Vol path (extension panel)
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.axhline(SIGMA * 100, color=C["blue"], lw=1.8, ls="--", label="constant sigma (AC)")
    ax2.plot(t_hrs, sigma_sv * 100, color=C["teal"], lw=2.0, label="stoch. sigma (ext.)")
    ax2.fill_between(t_hrs, sigma_sv * 100, SIGMA * 100,
                     where=(sigma_sv * 100 > SIGMA * 100),
                     alpha=0.18, color=C["coral"], label="vol spike")
    ax2.fill_between(t_hrs, sigma_sv * 100, SIGMA * 100,
                     where=(sigma_sv * 100 <= SIGMA * 100),
                     alpha=0.15, color=C["teal"])
    style(ax2, "Volatility path (extension)", "trading hour", "annualised vol (%)")
    ax2.legend(fontsize=7.5, framealpha=0)
    ax2.set_xlim(0, t_hrs[-1])

    # 3. Efficient frontier
    ax3 = fig.add_subplot(gs[0, 2])
    bps_cost = e_costs / (X0 * S0) * 1e4
    bps_std  = np.sqrt(e_vars) / (X0 * S0) * 1e4
    ax3.plot(bps_std, bps_cost, color=C["teal"], lw=2.2, label="efficient frontier")
    cur_e = ac.expected_cost() / (X0 * S0) * 1e4
    cur_s = np.sqrt(ac.cost_variance()) / (X0 * S0) * 1e4
    ax3.scatter([cur_s], [cur_e], color=C["coral"], s=80, zorder=5,
                label=f"current (lam={LAMBDA:.0e})")
    style(ax3, "Efficient frontier (AC)", "IS std dev (bps)", "expected IS (bps)")
    ax3.legend(fontsize=7.5, framealpha=0)

    # 4. Trade schedule bar chart
    ax4 = fig.add_subplot(gs[1, 0])
    trades = ac.trade_list()
    ax4.bar(np.arange(1, N + 1), trades / 1e3, color=C["blue"], alpha=0.78, width=0.7)
    style(ax4, "Trade schedule", "slice index", "shares sold (k)")

    # 5. Impact decomposition
    ax5 = fig.add_subplot(gs[1, 1])
    slices = np.arange(1, N + 1)
    ax5.bar(slices, perm_bps, label="permanent",  color=C["blue"],  alpha=0.85)
    ax5.bar(slices, temp_bps, label="temporary",  color=C["teal"],  alpha=0.85,
            bottom=perm_bps)
    style(ax5, "Impact decomposition (bps/slice)", "slice index", "cost (bps)")
    ax5.legend(fontsize=7.5, framealpha=0)

    # 6. IS cost distribution
    ax6 = fig.add_subplot(gs[1, 2])
    ac_bps = mc_costs_all / (X0 * S0) * 1e4
    sv_bps = sv_costs_all / (X0 * S0) * 1e4
    lo = min(ac_bps.min(), sv_bps.min())
    hi = max(ac_bps.max(), sv_bps.max())
    bins = np.linspace(lo, hi, 42)
    ax6.hist(ac_bps, bins=bins, color=C["blue"],  alpha=0.6,
             label=f"AC  mu={ac_bps.mean():.1f} bps")
    ax6.hist(sv_bps, bins=bins, color=C["coral"], alpha=0.6,
             label=f"Stoch-vol mu={sv_bps.mean():.1f} bps")
    style(ax6, "IS cost distribution (MC)", "IS (bps)", "frequency")
    ax6.legend(fontsize=7.5, framealpha=0)

    # 7. Lambda sensitivity
    ax7 = fig.add_subplot(gs[2, 0])
    lam_sweep = np.logspace(-9, -4, 60)
    kappas_sweep = []
    for l in lam_sweep:
        m = AlmgrenChriss(lam=l)
        kappas_sweep.append(m.kappa)
    ax7.semilogx(lam_sweep, kappas_sweep, color=C["purple"], lw=2.2)
    ax7.axvline(LAMBDA, color=C["coral"], ls="--", lw=1.5,
                label=f"lambda={LAMBDA:.0e}")
    style(ax7, "Urgency kappa vs risk aversion", "lambda (log scale)", "kappa")
    ax7.legend(fontsize=7.5, framealpha=0)

    # 8. Adaptive kappa in SV model
    ax8 = fig.add_subplot(gs[2, 1])
    sig_range = np.linspace(0.05, 0.80, 200)
    kap_range = []
    for s in sig_range:
        ss = s * S0 * np.sqrt(DT)
        kap_range.append(np.sqrt(LAMBDA * ss**2 / ETA))
    ax8.plot(sig_range * 100, kap_range, color=C["amber"], lw=2.2)
    ax8.axvline(SIGMA * 100, color=C["blue"], ls="--", lw=1.4, label="base vol")
    style(ax8, "Adaptive urgency: kappa vs vol", "instantaneous vol (%)", "kappa")
    ax8.legend(fontsize=7.5, framealpha=0)
    ax8.text(35, max(kap_range)*0.6,
             "vol spikes force\nfaster execution", fontsize=8,
             color=C["gray"], ha="left")

    # 9. Vol regime comparison
    ax9 = fig.add_subplot(gs[2, 2])
    regimes = {"10% vol": 0.10, "25% vol (base)": 0.25, "50% vol": 0.50}
    cols = [C["teal"], C["blue"], C["coral"]]
    for (label, sig), col in zip(regimes.items(), cols):
        m = AlmgrenChriss(sigma=sig)
        _, x = m.trajectory()
        ax9.plot(t_hrs, x / 1e6, color=col, lw=2.0, label=label)
    style(ax9, "Vol regime effect on schedule", "trading hour", "shares remaining (M)")
    ax9.legend(fontsize=7.5, framealpha=0)
    ax9.set_xlim(0, t_hrs[-1])
    ax9.set_ylim(0, 1.05)

    fig.text(0.01, 0.005,
             "Part 1 (blue): closed-form AC replication   |   "
             "Part 2 (teal/coral): stochastic vol extension -- relaxes constant-sigma via CIR mean-reverting variance process",
             fontsize=7.5, color=C["gray"], style="italic")

    plt.close()
    plt.savefig("almgren_chriss_summary.png", dpi=300)


#  SUMMARY PRINTOUT


def print_summary():
    ac  = AlmgrenChriss()
    svc = StochasticVolAC(xi=0.4, kappa_v=8.0, seed=99)

    e_cost   = ac.expected_cost()
    e_var    = ac.cost_variance()
    kappa    = ac.kappa
    hl_hrs   = np.log(2) / kappa * 252 * 6.5 if kappa > 1e-8 else float("inf")
    mc_costs = ac.simulate_paths(n_paths=1000)
    _, _, sv_costs = svc.simulate(n_paths=1000)

    def bps(dollars):
        return dollars / (X0 * S0) * 1e4

    
    print("  ALMGREN-CHRISS MODEL (Part 1)")
    print(f"  Position          : {X0:,} shares @ ${S0:.2f}  = ${X0*S0/1e6:.0f}M")
    print(f"  Horizon           : 1 trading day  ({T_YR*252*6.5:.1f} hrs)")
    print(f"  Risk aversion     : {LAMBDA:.1e}")
    print(f"  Annual volatility : {SIGMA*100:.0f}%")
    print(f"  kappa             : {kappa:.5f}")
    print(f"  Trade half-life   : {hl_hrs:.2f} hrs")
    print(f"  E[IS] analytical  : ${e_cost:,.0f}  ({bps(e_cost):.2f} bps)")
    print(f"  IS std dev        : ${np.sqrt(e_var):,.0f}  ({bps(np.sqrt(e_var)):.2f} bps)")
    print(f"  MC mean IS        : ${mc_costs.mean():,.0f}  ({bps(mc_costs.mean()):.2f} bps)")
    print("  STOCHASTIC VOL EXTENSION (Part 2)")
    print(f"  SV mean IS        : ${sv_costs.mean():,.0f}  ({bps(sv_costs.mean()):.2f} bps)")
    print(f"  SV IS std dev     : ${sv_costs.std():,.0f}  ({bps(sv_costs.std()):.2f} bps)")
    wider = (sv_costs.std() - np.sqrt(e_var)) / np.sqrt(e_var) * 100
    print(f"  Dist. wider vs AC : +{wider:.1f}%  (unmodelled vol risk)")
    print()
    print("  Extension insight: The stochastic vol model produces")
    print("  a wider IS distribution because unhedged vol spikes")
    print("  force suboptimal speedups. AC understates tail risk.")
    print()


if __name__ == "__main__":
    print_summary()
    plot_all()
    

  ALMGREN-CHRISS MODEL (Part 1)
  Position          : 1,000,000 shares @ $50.00  = $50M
  Horizon           : 1 trading day  (6.5 hrs)
  Risk aversion     : 1.0e-06
  Annual volatility : 25%
  kappa             : 10.20653
  Trade half-life   : 111.24 hrs
  E[IS] analytical  : $75,013  (15.00 bps)
  IS std dev        : $27,555  (5.51 bps)
  MC mean IS        : $89,390  (17.88 bps)
  STOCHASTIC VOL EXTENSION (Part 2)
  SV mean IS        : $77,496  (15.50 bps)
  SV IS std dev     : $465,076  (93.02 bps)
  Dist. wider vs AC : +1587.8%  (unmodelled vol risk)

  Extension insight: The stochastic vol model produces
  a wider IS distribution because unhedged vol spikes
  force suboptimal speedups. AC understates tail risk.

Generating 9-panel figure ...
Done.


In [5]:
"""
Almgren-Chriss Optimal Execution Model (Dark Mode Edition)
==========================================================
Includes closed-form AC replication and Stochastic Volatility extension.
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# GLOBAL PARAMETERS
X0      = 1_000_000          # total shares to liquidate
S0      = 50.0               # arrival mid-price ($)
ETA     = 2.976e-10          # temporary impact (calibrated for ~15 bps)
GAMMA   = 1.5e-11            # permanent impact
SIGMA   = 0.25               # annual volatility (constant, AC base case)
T_YR    = 1 / 252            # horizon: 1 trading day expressed in years
N       = 20                 # number of equal time slices
DT      = T_YR / N
LAMBDA  = 1e-6               # risk aversion parameter


# PART 1 -- ORIGINAL ALMGREN-CHRISS MODEL

class AlmgrenChriss:
    def __init__(self, X0=X0, S0=S0, eta=ETA, gamma=GAMMA,
                 sigma=SIGMA, T=T_YR, N=N, lam=LAMBDA):
        self.X0    = X0
        self.S0    = S0
        self.eta   = eta
        self.gamma = gamma
        self.T     = T
        self.N     = N
        self.dt    = T / N
        self.lam   = lam
        self.sigma_step = sigma * S0 * np.sqrt(self.dt)
        self.kappa      = np.sqrt(lam * self.sigma_step**2 / eta)

    def trajectory(self):
        t     = np.linspace(0, self.T, self.N + 1)
        denom = np.sinh(self.kappa * self.T)
        if denom < 1e-12:
            x = self.X0 * (1 - t / self.T)
        else:
            x = self.X0 * np.sinh(self.kappa * (self.T - t)) / denom
        return t, np.maximum(x, 0.0)

    def trade_list(self):
        _, x = self.trajectory()
        return np.diff(-x)

    def expected_cost(self):
        k, T, X0, eta, gamma = self.kappa, self.T, self.X0, self.eta, self.gamma
        if k * T < 1e-8:
            return 0.5 * gamma * X0**2 + eta * X0**2 / T
        return (0.5 * gamma * X0**2
                + eta * X0**2 * k / (2 * np.tanh(k * T / 2)))

    def cost_variance(self):
        k, T, X0 = self.kappa, self.T, self.X0
        s2       = self.sigma_step**2
        if k * T < 1e-8:
            return s2 * X0**2 * T / 3
        return (s2 * X0**2
                * (k * T / 2 - 0.5 * np.tanh(k * T / 2))
                / (k * np.sinh(k * T)))

    def simulate_paths(self, n_paths=500, seed=42):
        rng    = np.random.default_rng(seed)
        trades = self.trade_list()
        pnl    = np.zeros(n_paths)
        for p in range(n_paths):
            price    = self.S0
            total    = 0.0
            for j, n in enumerate(trades):
                dW         = rng.standard_normal() * self.sigma_step
                price     += self.gamma * n + dW
                exec_px    = price + self.eta * n / self.dt
                total     += n * exec_px
            pnl[p] = total - self.X0 * self.S0
        return pnl

    def efficient_frontier(self, n_points=80):
        lam_grid = np.logspace(-9, -4, n_points)
        costs, variances = [], []
        for l in lam_grid:
            m = AlmgrenChriss(self.X0, self.S0, self.eta, self.gamma,
                              SIGMA, self.T, self.N, l)
            costs.append(m.expected_cost())
            variances.append(m.cost_variance())
        return np.array(costs), np.array(variances)

    def impact_decomposition(self):
        trades = self.trade_list()
        perm   = self.gamma * trades / self.S0 * 1e4
        temp   = self.eta * (trades / self.dt) / self.S0 * 1e4
        return perm, temp


# PART 2 -- EXTENSION: STOCHASTIC VOLATILITY

class StochasticVolAC:
    def __init__(self, X0=X0, S0=S0, eta=ETA, gamma=GAMMA,
                 T=T_YR, N=N, lam=LAMBDA,
                 kappa_v=8.0, theta_v=None, xi=0.4, seed=42):
        self.X0      = X0
        self.S0      = S0
        self.eta     = eta
        self.gamma   = gamma
        self.T       = T
        self.N       = N
        self.dt      = T / N
        self.lam     = lam
        self.kappa_v = kappa_v
        self.theta_v = theta_v if theta_v is not None else SIGMA**2
        self.xi      = xi
        self.V0      = SIGMA**2
        self.rng     = np.random.default_rng(seed)

    def _local_kappa(self, V):
        sigma_step = np.sqrt(V) * self.S0 * np.sqrt(self.dt)
        return np.sqrt(max(self.lam * sigma_step**2 / self.eta, 0.0))

    def _local_trade(self, x_rem, V, steps_left):
        if steps_left <= 1:
            return x_rem
        T_rem = steps_left * self.dt
        k     = self._local_kappa(V)
        if k * T_rem < 1e-8:
            return x_rem / steps_left
        n = x_rem * (1 - np.sinh(k * (T_rem - self.dt)) / np.sinh(k * T_rem))
        return max(float(n), 0.0)

    def _step_variance(self, V):
        dW = self.rng.standard_normal()
        dV = (self.kappa_v * (self.theta_v - V) * self.dt
              + self.xi * np.sqrt(max(V, 0) * self.dt) * dW)
        return max(V + dV, 1e-12)

    def simulate(self, n_paths=500):
        trajectories = np.zeros((n_paths, self.N + 1))
        vol_paths    = np.zeros((n_paths, self.N + 1))
        costs        = np.zeros(n_paths)

        for p in range(n_paths):
            x = float(self.X0)
            V = self.V0
            price = self.S0
            trajectories[p, 0] = x
            vol_paths[p, 0]    = np.sqrt(V)

            for j in range(self.N):
                n = min(self._local_trade(x, V, self.N - j), x)
                sigma_step = np.sqrt(V) * self.S0 * np.sqrt(self.dt)
                dW         = self.rng.standard_normal() * sigma_step
                price     += self.gamma * n + dW
                exec_px    = price + self.eta * n / self.dt
                costs[p]  += n * exec_px
                x         -= n
                V          = self._step_variance(V)
                trajectories[p, j+1] = x
                vol_paths[p, j+1]    = np.sqrt(V)

            if x > 0:
                costs[p] += x * price

        costs -= self.X0 * self.S0
        return trajectories, vol_paths, costs

    def mean_trajectory(self, n_paths=500):
        trajs, vols, costs = self.simulate(n_paths)
        return trajs.mean(0), vols.mean(0), costs


# VISUALIZATION - DARK MODE

C = {
    "bg"     : "#121212",  # Deep charcoal background
    "panel"  : "#1E1E1E",  # Slightly lighter panel background
    "text"   : "#E0E0E0",  # Off-white for text
    "grid"   : "#333333",  # Subtle grid lines
    "blue"   : "#4DA8DA",  # Bright neon blue
    "teal"   : "#20C997",  # Neon teal
    "coral"  : "#FF6B6B",  # Bright coral/pink
    "amber"  : "#FCA311",  # Vibrant amber
    "gray"   : "#888888",  # Medium gray
    "purple" : "#B19CD9",  # Pastel purple
    "red"    : "#FF4D4D",  # Bright red
}

def plot_all():
    plt.style.use('dark_background')
    
    ac  = AlmgrenChriss()
    svc = StochasticVolAC(xi=0.4, kappa_v=8.0, seed=7)

    t_grid = np.linspace(0, T_YR, N + 1)
    t_hrs  = t_grid * 252 * 6.5

    _, x_ac   = ac.trajectory()
    x_twap    = X0 * (1 - np.linspace(0, 1, N + 1))
    x_sv, sigma_sv, sv_costs = svc.mean_trajectory(n_paths=800)

    perm_bps, temp_bps = ac.impact_decomposition()
    e_costs, e_vars    = ac.efficient_frontier()

    mc_costs_all = ac.simulate_paths(n_paths=600)
    _, _, sv_costs_all = svc.simulate(n_paths=600)

    fig = plt.figure(figsize=(18, 14), facecolor=C["bg"])
    fig.suptitle(
        "Almgren-Chriss Optimal Execution  |  Replication + Stochastic Vol Extension",
        fontsize=16, fontweight="bold", y=0.97, color=C["text"]
    )

    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

    def style(ax, title, xlabel, ylabel):
        ax.set_facecolor(C["panel"])
        ax.set_title(title, fontsize=12, fontweight="bold", pad=10, color=C["text"])
        ax.set_xlabel(xlabel, fontsize=10, color=C["text"])
        ax.set_ylabel(ylabel, fontsize=10, color=C["text"])
        ax.tick_params(colors=C["text"], labelsize=9)
        for spine in ax.spines.values():
            spine.set_color(C["grid"])
        ax.grid(True, color=C["grid"], alpha=0.6, lw=0.8, ls="--")
        
    def add_legend(ax):
        # Creates a clean legend with a solid background so lines don't bleed through
        ax.legend(fontsize=9, facecolor=C["panel"], edgecolor=C["grid"], 
                  labelcolor=C["text"], framealpha=0.9, loc='best')

    # 1. Execution trajectories
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(t_hrs, x_ac / 1e6,   color=C["blue"],   lw=2.5, label="AC optimal")
    ax1.plot(t_hrs, x_twap / 1e6, color=C["gray"],   lw=1.5, ls="--", label="TWAP")
    ax1.plot(t_hrs, x_sv / 1e6,   color=C["teal"],   lw=2.5, ls="-.", label="Stoch-vol AC")
    style(ax1, "Execution Trajectory", "Trading Hour", "Shares Remaining (M)")
    add_legend(ax1)
    ax1.set_xlim(0, t_hrs[-1])
    ax1.set_ylim(0, 1.05)

    # 2. Vol path (extension panel)
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.axhline(SIGMA * 100, color=C["blue"], lw=2.0, ls="--", label="Constant Sigma (AC)")
    ax2.plot(t_hrs, sigma_sv * 100, color=C["teal"], lw=2.5, label="Stoch. Sigma (Ext.)")
    ax2.fill_between(t_hrs, sigma_sv * 100, SIGMA * 100,
                     where=(sigma_sv * 100 > SIGMA * 100),
                     alpha=0.25, color=C["coral"], label="Vol Spike")
    ax2.fill_between(t_hrs, sigma_sv * 100, SIGMA * 100,
                     where=(sigma_sv * 100 <= SIGMA * 100),
                     alpha=0.15, color=C["teal"])
    style(ax2, "Volatility Path (Extension)", "Trading Hour", "Annualised Vol (%)")
    add_legend(ax2)
    ax2.set_xlim(0, t_hrs[-1])

    # 3. Efficient frontier
    ax3 = fig.add_subplot(gs[0, 2])
    bps_cost = e_costs / (X0 * S0) * 1e4
    bps_std  = np.sqrt(e_vars) / (X0 * S0) * 1e4
    ax3.plot(bps_std, bps_cost, color=C["teal"], lw=2.5, label="Efficient Frontier")
    cur_e = ac.expected_cost() / (X0 * S0) * 1e4
    cur_s = np.sqrt(ac.cost_variance()) / (X0 * S0) * 1e4
    ax3.scatter([cur_s], [cur_e], color=C["coral"], s=100, zorder=5,
                label=f"Current ($\lambda$={LAMBDA:.0e})")
    style(ax3, "Efficient Frontier (AC)", "IS Std Dev (bps)", "Expected IS (bps)")
    add_legend(ax3)

    # 4. Trade schedule bar chart
    ax4 = fig.add_subplot(gs[1, 0])
    trades = ac.trade_list()
    ax4.bar(np.arange(1, N + 1), trades / 1e3, color=C["blue"], alpha=0.85, width=0.7)
    style(ax4, "Trade Schedule", "Slice Index", "Shares Sold (k)")

    # 5. Impact decomposition
    ax5 = fig.add_subplot(gs[1, 1])
    slices = np.arange(1, N + 1)
    ax5.bar(slices, perm_bps, label="Permanent",  color=C["blue"],  alpha=0.9)
    ax5.bar(slices, temp_bps, label="Temporary",  color=C["teal"],  alpha=0.9,
            bottom=perm_bps)
    style(ax5, "Impact Decomposition", "Slice Index", "Cost (bps)")
    add_legend(ax5)

    # 6. IS cost distribution
    ax6 = fig.add_subplot(gs[1, 2])
    ac_bps = mc_costs_all / (X0 * S0) * 1e4
    sv_bps = sv_costs_all / (X0 * S0) * 1e4
    lo = min(ac_bps.min(), sv_bps.min())
    hi = max(ac_bps.max(), sv_bps.max())
    bins = np.linspace(lo, hi, 42)
    ax6.hist(ac_bps, bins=bins, color=C["blue"],  alpha=0.7,
             label=f"AC ($\mu$={ac_bps.mean():.1f} bps)")
    ax6.hist(sv_bps, bins=bins, color=C["coral"], alpha=0.7,
             label=f"Stoch-Vol ($\mu$={sv_bps.mean():.1f} bps)")
    style(ax6, "IS Cost Distribution (MC)", "Implementation Shortfall (bps)", "Frequency")
    add_legend(ax6)

    # 7. Lambda sensitivity
    ax7 = fig.add_subplot(gs[2, 0])
    lam_sweep = np.logspace(-9, -4, 60)
    kappas_sweep = []
    for l in lam_sweep:
        m = AlmgrenChriss(lam=l)
        kappas_sweep.append(m.kappa)
    ax7.semilogx(lam_sweep, kappas_sweep, color=C["purple"], lw=2.5)
    ax7.axvline(LAMBDA, color=C["coral"], ls="--", lw=2.0,
                label=f"$\lambda$={LAMBDA:.0e}")
    style(ax7, "Urgency ($\kappa$) vs Risk Aversion", "Risk Aversion ($\lambda$)", "Urgency ($\kappa$)")
    add_legend(ax7)

    # 8. Adaptive kappa in SV model
    ax8 = fig.add_subplot(gs[2, 1])
    sig_range = np.linspace(0.05, 0.80, 200)
    kap_range = []
    for s in sig_range:
        ss = s * S0 * np.sqrt(DT)
        kap_range.append(np.sqrt(LAMBDA * ss**2 / ETA))
    ax8.plot(sig_range * 100, kap_range, color=C["amber"], lw=2.5)
    ax8.axvline(SIGMA * 100, color=C["blue"], ls="--", lw=2.0, label="Base Volatility")
    style(ax8, "Adaptive Urgency: $\kappa$ vs Vol", "Instantaneous Vol (%)", "Urgency ($\kappa$)")
    add_legend(ax8)
    ax8.text(35, max(kap_range)*0.6,
             "Vol spikes force\nfaster execution", fontsize=10,
             color=C["text"], ha="left", bbox=dict(facecolor=C["panel"], edgecolor=C["grid"], alpha=0.8))

    # 9. Vol regime comparison
    ax9 = fig.add_subplot(gs[2, 2])
    regimes = {"10% Vol": 0.10, "25% Vol (Base)": 0.25, "50% Vol": 0.50}
    cols = [C["teal"], C["blue"], C["coral"]]
    for (label, sig), col in zip(regimes.items(), cols):
        m = AlmgrenChriss(sigma=sig)
        _, x = m.trajectory()
        ax9.plot(t_hrs, x / 1e6, color=col, lw=2.5, label=label)
    style(ax9, "Vol Regime Effect on Schedule", "Trading Hour", "Shares Remaining (M)")
    add_legend(ax9)
    ax9.set_xlim(0, t_hrs[-1])
    ax9.set_ylim(0, 1.05)


    plt.savefig("almgren_chriss_dark_mode.png", dpi=300, bbox_inches='tight', facecolor=C["bg"])
    plt.close()

if __name__ == "__main__":
    plot_all()